In [4]:
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
# UserProxyAgent를 사용해서 더 깊은 research나 더 깊은 insight를 요청한다. 
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.ui import Console
from tools import web_search_tool, save_report_to_md

In [5]:
model_client = OpenAIChatCompletionClient(
    model="gpt-4.1-nano-2025-04-14",
)

In [7]:
research_planner = AssistantAgent(
    "research_planner",
    description="복잡한 질문을 연구 하위 과제로 분해하는 전략적 리서치 코디네이터",
    model_client=model_client,
    system_message="""당신은 리서치 기획 전문가입니다. 당신의 임무는 명확하고 집중된 리서치 계획을 수립하는 것입니다.

각 연구 질문에 대해 다음을 포함한 **집중도 높은 리서치 계획**을 작성하세요:

1. **핵심 주제(Core Topics)**: 조사해야 할 주요 영역 2~3개
2. **검색 쿼리(Search Queries)**: 다음을 포괄하는 구체적인 검색 쿼리 3~5개
   - 최신 동향 및 뉴스
   - 핵심 통계 또는 데이터
   - 전문가 분석 또는 연구 결과
   - 향후 전망

계획은 현실적이고 실행 가능해야 합니다.
양보다 질을 우선하세요.""",
)

research_agent = AssistantAgent(
    "research_agent",
    description="웹 검색을 수행하고 콘텐츠를 추출하는 리서치 전문가",
    tools=[web_search_tool],
    model_client=model_client,
    system_message="""당신은 웹 리서치 전문가입니다. 리서치 계획에 따라 집중적인 검색을 수행하세요.

리서치 전략:
1. 리서치 계획에 기반하여 **3~5개의 검색을 실행**하세요.
2. 검색 결과에서 다음 핵심 정보를 추출하세요:
   - 주요 사실 및 통계
   - 최근 동향
   - 전문가 의견
   - 중요한 맥락 정보

3. **품질 중심 원칙**:
   - 신뢰할 수 있는 권위 있는 출처를 우선하세요
   - 최근 정보(최근 2년 이내)를 중점적으로 찾으세요
   - 다양한 관점을 기록하세요

계획된 검색을 모두 완료한 후, 조사 결과를 요약하세요.
목표는 **5~10개의 고품질 출처**를 확보하는 것입니다.""",
)

research_analyst = AssistantAgent(
    "research_analyst",
    description="리서치 보고서를 작성하는 전문 분석가",
    model_client=model_client,
    system_message="""당신은 리서치 분석가입니다. 수집된 자료를 바탕으로 종합적인 리서치 보고서를 작성하세요.

다음 구조를 갖춘 **리서치 보고서**를 작성하세요:

## Executive Summary
- 핵심 발견 사항 및 결론
- 주요 인사이트

## Background & Current State
- 현재 시장/환경 개요
- 최근 동향
- 주요 통계 및 데이터

## Analysis & Insights
- 주요 트렌드
- 다양한 관점
- 전문가 의견

## Future Outlook
- 떠오르는 트렌드
- 전망 및 예측
- 시사점

## Sources
- 사용한 모든 출처 목록

수집된 리서치를 기반으로 명확하고 구조적인 보고서를 작성하세요.
작성이 끝나면 반드시 마지막에 "REPORT_COMPLETE"를 추가하세요.""",
)

quality_reviewer = AssistantAgent(
    "quality_reviewer",
    description="리서치의 완성도와 정확성을 검증하는 품질 관리 전문가",
    tools=[save_report_to_md],
    model_client=model_client,
    system_message="""당신은 품질 검토자입니다. 리서치 분석가가 완전한 리서치 보고서를 작성했는지 확인하세요.

다음을 확인하세요:
- 리서치 분석가가 작성한 종합적인 보고서이며, 마지막에 "REPORT_COMPLETE"가 포함되어 있는지
- 연구 질문이 충분히 답변되었는지
- 출처가 명시되어 있고 신뢰할 수 있는지
- 요약, 핵심 정보, 분석, 출처가 모두 포함되어 있는지

"REPORT_COMPLETE"로 끝나는 완전한 리서치 보고서를 확인하면:
1. 먼저 save_report_to_md 도구를 사용해 보고서를 report.md로 저장하세요.
2. 그 다음 다음 문장을 출력하세요:
   "The research is complete. The report has been saved to report.md. Please review the report and let me know if you approve it or need additional research."

아직 완전한 보고서가 작성되지 않았다면,
리서치 분석가에게 지금 보고서를 작성하라고 지시하세요.""",
)

research_enhancer = AssistantAgent(
    "research_enhancer",
    description="치명적인 리서치 공백만 식별하는 전문가",
    model_client=model_client,
    system_message="""당신은 리서치 보완 전문가입니다. **오직 치명적인 공백만** 식별하세요.

리서치를 검토한 뒤, 다음과 같은 **중대한 누락**이 있을 때만 추가 검색을 제안하세요:
- 최근 6개월 이내의 핵심 동향이 완전히 누락된 경우
- 통계나 데이터가 전혀 없는 경우
- 요청된 중요한 관점이 빠진 경우

기본적인 내용이 충분히 잘 다뤄졌다면 다음과 같이 답하세요:
"The research is sufficient to proceed with the report."

정말 필요한 경우에만 **1~2개의 추가 검색**을 제안하세요.
완벽한 커버리지보다, 좋은 보고서를 완성하는 것을 우선합니다.""",
)

user_proxy = UserProxyAgent(
    "user_proxy",
    description="추가 리서치를 요청하거나 최종 결과를 승인하는 인간 검토자",
    input_func=input,
)

In [8]:
# 모든게 어떻게 작동해야 하는지 하나하나 적어야함 누가 언제 선택되고 언제 사용자에게 승인을 요청할지 언제 요청하지 않을지 등등 

selector_prompt = """
당신은 대화 흐름을 제어하는 관리자입니다. 
아래의 [워크플로우 규칙]을 **엄격히** 따르세요. 감정을 섞지 말고 규칙에 맞는 다음 발언자 이름만 출력하세요.

{roles}

현재 대화 내용:
{history}

사용 가능한 에이전트:
- research_planner: 리서치 접근 방식 기획 (초기 단계에서만 사용)
- research_agent: 웹 소스 검색 및 콘텐츠 추출 (기획 이후)
- research_enhancer: 치명적인 공백만 식별 (필요할 때만 제한적으로 사용)
- research_analyst: 최종 리서치 보고서 작성
- quality_reviewer: 완성된 보고서 존재 여부 및 품질 확인
- user_proxy: 인간 사용자에게 피드백 요청

작업 흐름(WORKFLOW):
1. 아직 리서치 기획이 없다면 → research_planner 선택
2. 기획은 완료되었으나 리서치가 없다면 → research_agent 선택
3. research_agent가 초기 검색을 완료한 후 → research_enhancer를 **1회만** 선택
4. enhancer가 "sufficient to proceed"라고 판단하면 → research_analyst 선택
5. enhancer가 치명적인 추가 검색을 제안하면 → research_agent를 **1회 추가 실행 후** research_analyst 선택
6. research_analyst가 "REPORT_COMPLETE"를 출력했다면 → quality_reviewer 선택
7. quality_reviewer가 사용자 피드백을 요청하면 → user_proxy 선택

중요:
research_agent는 **최대 2회까지만** 검색을 수행할 수 있습니다.
2회 검색이 완료되면 결과와 관계없이 research_analyst로 진행하세요.

위 작업 흐름에 따라 다음에 작업해야 할 에이전트를 선택하세요.
"""


In [9]:
text_termination = TextMentionTermination("APPROVED")
max_message_termination = MaxMessageTermination(max_messages=50)
termination_condition = text_termination | max_message_termination

team = SelectorGroupChat(
    participants=[
        research_agent,
        research_analyst,
        research_enhancer,
        research_planner,
        quality_reviewer,
        user_proxy,
    ],
    selector_prompt=selector_prompt,
    # model_client에게 전달해야 하는 이유는 selectorGroupChat은 자기만의 AI를 가지고 participant 중에서 선택해야 함. 그래서 전달
    model_client=model_client,
    # allow_repeated_speaker=True, #한 agent가 여러번 말하고 반복할 수 있다는 뜻.
    termination_condition=termination_condition,
)

In [1]:
await Console(team.run_stream(task="원자력 에너지의 최신 기술 및 개발 동향을 조사하세요"))

NameError: name 'Console' is not defined